# Digit Recognizer — *casa Zaccaria* notebook

Goal of this notebook is **learning**, not chasing the leaderboard: it is the CNN
dojo before moving on to real OCR/HTR tasks. We build a small, honest pipeline and
grow it version by version (v0 → v1 → v2 …), each step with a **pre-declared success
criterion** and a logged **ADOPT / REJECT** verdict.

**House rules applied here**
- The out-of-fold (OOF) score is the *authoritative* signal. The public leaderboard is noise.
- Metric alignment: our OOF metric (accuracy) matches the competition metric before we trust anything.
- Validation scheme chosen deliberately (see the EDA section) — not copy-pasted.
- A valid submission is produced on day one, to de-risk the plumbing early.
- The experiment log lives *inside* this notebook from the first commit and is the skeleton of the write-up.

## 1. Setup

Reproducible seeds and a single place for configuration. On Kaggle, set the notebook
accelerator to **GPU** (Settings → Accelerator) before running the CNN section.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5
DATA_DIR = "../input/digit-recognizer"   # Kaggle default mount path

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 50)
print("Setup done. Seed:", SEED)

## 2. Load data

The training file is 785 columns: one `label` plus 784 flattened pixels (28×28).
Pixels are 0–255; we scale to [0, 1]. Scaling is not "cleaning" — it just helps the
optimizer converge.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

y = train["label"].values
X = train.drop(columns="label").values.astype("float32") / 255.0   # (n, 784)
X_test = test.values.astype("float32") / 255.0                      # (m, 784)

print("train:", X.shape, " test:", X_test.shape)
print("labels:", np.unique(y))

## 3. Quick EDA — *and the validation decision*

Two questions only, both of which drive the validation scheme:

1. **Are the classes balanced?** If yes, plain accuracy is an honest metric and we do
   not need any imbalance machinery (prior-correction, calibration, etc.).
2. **Are the samples independent?** In *Trace the Ace* multiple samples shared a
   `session_id`, which forced `GroupKFold`. Here every image is an independent digit —
   there is no grouping key — so the correct choice is `StratifiedKFold` (stratified on
   the label, to keep each fold's class mix representative).

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(y).value_counts().sort_index()
print(counts)
print("min/max class share:", round(counts.min()/len(y), 4), "/", round(counts.max()/len(y), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(counts.index, counts.values)
ax[0].set_title("Class balance")
ax[0].set_xlabel("digit"); ax[0].set_ylabel("count"); ax[0].set_xticks(range(10))

# One example per class, to eyeball what we are actually classifying.
grid = np.zeros((28, 28*10))
for d in range(10):
    img = X[y == d][0].reshape(28, 28)
    grid[:, d*28:(d+1)*28] = img
ax[1].imshow(grid, cmap="gray_r")
ax[1].set_title("One sample per class")
ax[1].axis("off")
plt.tight_layout(); plt.show()

**Decision (logged):** classes are ~balanced (each digit ≈ 10% of the data), so
accuracy is a fair metric and no imbalance correction is warranted. Samples are
independent, so the validation scheme is `StratifiedKFold(n_splits=5, shuffle=True)`.
This same split object is reused across every version so that OOF scores are directly
comparable version-to-version (apples to apples).

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def write_submission(pred, filename):
    # ImageId is 1-indexed per the competition format.
    sub = pd.DataFrame({"ImageId": np.arange(1, len(pred) + 1), "Label": pred})
    sub.to_csv(filename, index=False)
    print(f"{filename} written:", len(sub), "rows")
    return sub

## 4. v0 — logistic regression baseline

The floor. A linear model on raw pixels can only draw straight decision boundaries in
784-dimensional pixel space, and it treats pixel (5,5) and (5,6) as unrelated columns —
no notion of *where* things are in the image. Whatever number it reaches is the ceiling
of the "an image is just 784 scalars" mindset. Everything after v0 has to earn its
complexity by beating this.

**Pre-declared criterion:** none — v0 *is* the baseline. Expected around ~0.92.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

oof_v0 = cross_val_predict(
    LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1),
    X, y, cv=skf, method="predict", n_jobs=-1,
)
cv_v0 = accuracy_score(y, oof_v0)
print("v0 OOF accuracy:", round(cv_v0, 4))

In [ ]:
# Fit on all data and emit a valid submission on day one (de-risk the pipeline).
clf = LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1).fit(X, y)
_ = write_submission(clf.predict(X_test), "submission_v0.csv")

## 5. v1 — the first CNN

Now we stop pretending. We reshape the 784 columns back into a 28×28×1 **image** and
let a convolutional network learn its own features via two ideas:

- **Locality** — small 3×3 filters look at one neighborhood at a time.
- **Weight sharing** — the *same* filter slides across the whole image, so a feature
  detector (say, a diagonal edge) works anywhere. This also buys translation tolerance,
  reinforced by `MaxPooling`, which keeps the strongest response per region and discards
  its exact position.

Kept deliberately minimal (no BatchNorm, no augmentation) so the v0 → v1 delta cleanly
isolates *"the model now knows it is looking at an image."* Those extras arrive in v2+.

**Pre-declared criterion:** v1 must beat v0's OOF accuracy. If it does not, it is a bug
to hunt, not a disappointment to accept.

In [ ]:
# Reshape flat pixels -> images. This is the whole conceptual jump from v0.
X_img = X.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)
print("image tensors:", X_img.shape, X_test_img.shape)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)


def build_cnn():
    # Minimal CNN: two conv blocks learn spatial features, then a classifier head.
    m = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),   # 32 filters slide over the image
        layers.MaxPooling2D(),                      # keep strongest response per region
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m


build_cnn().summary()

In [ ]:
# Same OOF discipline as v0, same split object -> scores are directly comparable.
oof_v1 = np.zeros(len(y), dtype="int64")
test_probs = np.zeros((len(X_test_img), 10), dtype="float32")

early = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn()
    model.fit(
        X_img[tr], y[tr],
        validation_data=(X_img[va], y[va]),
        epochs=20, batch_size=128, callbacks=[early], verbose=2,
    )
    oof_v1[va] = model.predict(X_img[va], verbose=0).argmax(axis=1)
    # Bag test predictions across the 5 fold-models -> more stable submission.
    test_probs += model.predict(X_test_img, verbose=0) / N_SPLITS
    print(f"fold {fold} accuracy:", round(accuracy_score(y[va], oof_v1[va]), 4))

cv_v1 = accuracy_score(y, oof_v1)
print("\nv1 OOF accuracy:", round(cv_v1, 4))
print("delta vs v0:", round(cv_v1 - cv_v0, 4))
print("criterion (beat v0):", "PASS" if cv_v1 > cv_v0 else "FAIL")

In [ ]:
# Bagged submission (average of the 5 fold-models). Honest model averaging, not an LB trick.
pred_v1 = test_probs.argmax(axis=1)
_ = write_submission(pred_v1, "submission_v1.csv")

### Sanity check — where does v1 still fail?

Shared errors across model families signal irreducible noise; a confusion matrix tells
us *which* digits get mixed up, which is exactly the kind of observation the write-up
lives on (e.g. 4↔9, 3↔5, 7↔1).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y, oof_v1)
disp = ConfusionMatrixDisplay(cm, display_labels=range(10))
fig, ax = plt.subplots(figsize=(5.5, 5.5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("v1 OOF confusion matrix")
plt.tight_layout(); plt.show()

## 6. Experiment log  *(the write-up skeleton)*

Filled top-to-bottom as we go. The criterion column is written **before** seeing the
result; the verdict column **after**.

| Version | Description | CV (OOF acc) | Pre-declared criterion | Verdict |
|--------:|-------------|:------------:|------------------------|:-------:|
| v0 | Logistic regression on raw pixels | 0.9207 | baseline floor | REFERENCE |
| v1 | Minimal CNN, 2 conv blocks, 5-fold bagged | *fill after run* | beat v0 (0.9207) | *ADOPT / REJECT* |
| v2 | v1 + data augmentation (shift / rotate / zoom) | — | beat v1 | *planned* |
| v3 | v2 + BatchNorm / dropout / deeper head | — | beat v2 | *planned* |

**Running notes**
- OOF ↔ LB agreement on v0 was near-perfect (0.9207 vs 0.9205): the pipeline is clean and
  the OOF number can be trusted to drive decisions.
- *(add one line per experiment: what changed, the delta, and the verdict.)*

## 7. Next steps

- **v2 — data augmentation.** Manufacture new training images by small random shifts,
  rotations and zooms so the network sees more variation than the fixed training set
  offers. On MNIST this is the standard next lever. Pre-declared criterion: beat v1 OOF.
- Later: BatchNorm + dropout, a slightly deeper head, and light Optuna tuning (always
  with the current best enqueued as an anti-regression trial).